# Cruce de mapas de riesgos con controles de ISOLUCION

### Importar librerías necesarias

In [1]:
import pandas as pd

### Limpiar datos

#### Mapa de riesgos consolidado (desde Excel)

##### Cargar data de mapa de riesgos consolidado

In [2]:
# Leer archivo
mapa_excel = pd.read_excel('Consolidado_2025_v13.xlsx', sheet_name = 'Mapa Riesgos')

/home/jairoescrito/miniconda3/envs/jairoescrito/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


##### Limpiar data de mapa de riesgos consolidado

In [3]:
# Seleccionar variables de inetrés
mapa_ini = mapa_excel.iloc[9:,[0,2,3,6,9,10,11,13,14,15,29,30,31,32,33]]
# Eliminar filas en blanco
mapa_ini.dropna(how='all', inplace=True)
# Resetear índices 
mapa_ini = mapa_ini.reset_index(drop=True)
# Eliminar filas definitivas
mapa_ini = mapa_ini.iloc[:-3, :]

In [4]:
# Extraer nombres de columnas
columnas_mapa_ini = mapa_ini.iloc[0,:]
columnas_mapa_ini = columnas_mapa_ini.reset_index().iloc[:,1]
# Ajustar texto
columnas_mapa_ini = (columnas_mapa_ini.astype(str)
                    .str.lower()
                    .str.replace(" ", "_")
                    .str.replace("\n", ""))

In [5]:
# Cambiar algunos textos
columnas_mapa_ini.iloc[5] = "%_probabilidad_inherente"
columnas_mapa_ini.iloc[8] = "%_impacto_inherente"
columnas_mapa_ini.iloc[11] = "%_probabilidad_residual"
columnas_mapa_ini.iloc[13] = "%_impacto_residual"

In [6]:
# Cambiar los nombres de las columnas
mapa_ini.columns = columnas_mapa_ini
mapa_ini = mapa_ini.iloc[1:,:]

In [7]:
# Eliminar caracteres de espacios en blanco que están de mas
# Seleccionar las columnas que son tipo object (texto)
cols_texto = mapa_ini.select_dtypes(include='object').columns
# Para cada columna reemplazar el \n por espacio y luego eliminar espacios al inicio y al final 
mapa_ini[cols_texto] = (mapa_ini[cols_texto]
    .apply(lambda col: col.where(
        col.isna(),
        col.astype(str).str.replace(r'\\n|\n', ' ', regex=True).str.strip()
    ))
)

In [8]:
# Llenar los NaN hacia abajo
mapa_ini = mapa_ini.ffill()
# Cada riesgo se repite varias veces (segun la cantidad de controles)
# La última fila de cada riesgo tiene el valor final de riesgo residual
# Se selecciona de cada grupo 
mapa_ini = mapa_ini.groupby('consecutivo', sort=False).last().reset_index()

In [15]:
# Convertir a números las columnas que si son numéricas
for col in mapa_ini.select_dtypes(include='object').columns:
    mapa_ini[col] = pd.to_numeric(mapa_ini[col], errors='coerce').fillna(mapa_ini[col])

#### Mapa de riesgos desde controles ISOLUCION

##### Cargar data de reporte de controles ISOLUCION

In [9]:
# Leer archivo
mapa_iso =  pd.read_excel('Reporte Mejoramiento Continuo - Controles.xlsx', 
                        sheet_name = 'Data')

##### Limpiar data de reporte de riesgos

In [10]:
# Seleccionar solo las columnas relativas a los riesgos
mapa_rep = mapa_iso.iloc[:, [1, 7, 9]].copy()

In [11]:
# Eliminar las filas en blanco
mapa_rep.dropna(how='all', inplace=True)
# Resetear índices 
mapa_rep = mapa_rep.reset_index(drop=True)

In [12]:
# Eliminar mayúsculas y acentos en los nombres de las columnas
mapa_rep.columns = (mapa_rep.columns
                    .str.lower()
                    .str.normalize('NFKD')
                    .str.encode('ascii', errors='ignore')
                    .str.decode('utf-8'))

In [13]:
# Cambiar número de identificación del riesgo a entero
mapa_rep['num'] = mapa_rep['num'].astype(int)

## Cruce de riesgos identificados